# Notebook hackathon : AssurPrime : Saurez-vous prédire la prime d'assurance ?
#### **Crédit Agricole Assurances**

lien hackathon : https://challengedata.ens.fr/participants/challenges/161/

PS : notebook Google Colab

## IMPORTATIONS ET INSTALLATIONS DES BIBLIOTHEQUES

In [1]:
pip install mlflow

  Using cached mlflow-3.11.1-py3-none-any.whl.metadata (49 kB)
  Using cached mlflow_skinny-3.11.1-py3-none-any.whl.metadata (49 kB)
  Using cached mlflow_tracing-3.11.1-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.2-py3-none-any.whl.metadata (5.3 kB)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached aiohttp-3.13.5-cp311-cp311-win_amd64.whl.metadata (8.4 kB)
  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached cryptography-46.0.7-cp311-abi3-win_amd64.whl.metadata (5.7 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached huey-2.6.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached pandas-2.3.3-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Using cached pyarrow-23.0.1-cp311-cp311-win_amd64.whl.metadata (3.1 kB)
  Using cached skops-0.13.0-py3-none-any.whl.metadata (5.6 kB)
  Using cached sqlalchemy-2.0.49-cp311-cp311-win_

In [2]:
pip install optuna


  Using cached optuna-4.8.0-py3-none-any.whl.metadata (17 kB)
  Using cached colorlog-6.10.1-py3-none-any.whl.metadata (11 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached optuna-4.8.0-py3-none-any.whl (419 kB)
Using cached colorlog-6.10.1-py3-none-any.whl (11 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)

   ---------------------------------------- 0/3 [tqdm]
   -------------------------- ------------- 2/3 [optuna]
   -------------------------- ------------- 2/3 [optuna]
   -------------------------- ------------- 2/3 [optuna]
   -------------------------- ------------- 2/3 [optuna]
   -------------------------- ------------- 2/3 [optuna]
   -------------------------- ------------- 2/3 [optuna]
   -------------------------- ------------- 2/3 [optuna]
   -------------------------- ------------- 2/3 [optuna]
   ---------------------------------------- 3/3 [optuna]



In [3]:
pip install optuna.integration

  Using cached optuna_integration-4.8.0-py3-none-any.whl.metadata (13 kB)
Using cached optuna_integration-4.8.0-py3-none-any.whl (103 kB)
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install catboost


  Using cached catboost-1.2.10-cp311-cp311-win_amd64.whl.metadata (1.5 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
  Using cached plotly-6.6.0-py3-none-any.whl.metadata (8.5 kB)
Using cached catboost-1.2.10-cp311-cp311-win_amd64.whl (100.2 MB)
Using cached graphviz-0.21-py3-none-any.whl (47 kB)
Using cached plotly-6.6.0-py3-none-any.whl (9.9 MB)

   ---------------------------------------- 0/4 [narwhals]
   ---------------------------------------- 0/4 [narwhals]
   ---------------------------------------- 0/4 [narwhals]
   ---------------------------------------- 0/4 [narwhals]
   ---------------------------------------- 0/4 [narwhals]
   ---------------------------------------- 0/4 [narwhals]
   ---------------------------------------- 0/4 [narwhals]
   ---------- ----------------------------- 1/4 [graphviz]
   -------------------- ------------------- 2/4 [plotly]
   -------------------- ------------------- 2/4 [plotly]
   -------------------- -----------------

In [5]:
pip install lightgbm

  Using cached lightgbm-4.6.0-py3-none-win_amd64.whl.metadata (17 kB)
Using cached lightgbm-4.6.0-py3-none-win_amd64.whl (1.5 MB)
Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install xgboost


  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)


In [9]:
import gc
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import lightgbm as lgb
from catboost import CatBoostRegressor, Pool
import xgboost as xgb

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import mean_squared_error, root_mean_squared_error

import optuna
from optuna.integration import LightGBMPruningCallback

import mlflow
import mlflow.lightgbm

In [ ]:
def charger_csv(path,fraction=0.20,chunksize=100000,graine=42):
    generateur = np.random.default_rng(graine)
    morceaux_ech = []
    for bloc in pd.read_csv(path,chunksize=chunksize):
        masque = generateur.random(len(bloc))
        morceaux_ech.append(bloc[masque])
    return pd.concat(morceaux_ech,ignore_index=True)

X_train_brut = charger_csv("train_input.csv",fraction=0.20,chunksize=100000,graine=42)
y_train_brut = charger_csv("train_output.csv",fraction=0.20,chunksize=100000,graine=42)

X_test_brut = charger_csv("test_input.csv",fraction=0.20,chunksize=100000,graine=42)


In [ ]:
def base_ingenierie(df):
    df_fe = df.copy()

    colonne_derog = [col for col in df_fe.columns if "DEROG" in col]
    colonne_caract = [col for col in df_fe.columns if "CARACT" in col]
    df_fe["NB_DEROG"] = df_fe[colonne_derog].notna().sum(axis=1).astype("int4").fillna(0)
    df_fe["NB_CARACT"] = df_fe[colonne_caract].notna().sum(axis=1).astype("int4").fillna(0)

    for col in "FRCH1" and "FRCH2" :
        df_fe[col] = (
            pd.to_numeric(df_fe["FRCH1"],errors="coerce").fillna(0),
            pd.to_numeric(df_fe["FRCH2"],errors="coerce").fillna(0)
        ).astype("float32")
    
    colonnes_meteo = [col for col in df_fe.columns if "NBJ" and "FFG" in col]
    if len(colonnes_meteo) > 0 :

        for col in colonnes_meteo :
            df_fe[col] = pd.to_numeric(df_fe[col],errors="coerce").astype("float32").fillna(0)

        df_fe["METEO_MAX"] = df_fe[colonnes_meteo].max(axis=1).astype("float32").fillna(0)
        df_fe["METEO_MOY"] = df_fe[colonnes_meteo].mean(axis=1).astype("float32").fillna(0)

    if col in "ANCIENNETE" :
        df_fe[col] = np.log1p(
            pd.to_numeric(df_fe["ANCIENNETE"],errors="coerce")
        ).astype("float3")

    return df_fe

In [ ]:
def reduire_taille(df):
    df_fe = df.copy()
    for col in df_fe.select_dtypes(include=["float32","float64"]).columns :
        df_fe[col] = pd.to_numeric(df_fe[col], downcast="float")
    for col in df_fe.select_dtypes(include=["int4","int8"]).columns :
        df_fe[col] = pd.to_numeric(df_fe[col], downcast="integer")
    return df_fe

In [ ]:
X_train_fe = reduire_taille(base_ingenierie(X_train_brut))
X_test_fe = reduire_taille(base_ingenierie(X_test_brut))

del X_train_brut
gc.collect()

In [ ]:
y_train_fe = y_train_brut.copy()
y_train_fe["CHARGE"] = y_train_fe["CHARGE"].clip(lower=0)
y_train_fe["PRIME_PURE"] = (y_train_fe["CHARGE"] / y_train_fe["ANNEE_ASSURANCE"]).astype("float32").fillna(0)
y_train_fe["A_SINISTRE"] = (y_train_fe["CHARGE"] > 0).astype(int)

In [ ]:
colonnes_supprimer = ["ID","ANNEE_ASSURANCE"]

variables = [col for col in X_train_fe.columns if col not in colonnes_supprimer]

colonnes_categorielles = (
    X_train_fe[variables].select_dtypes(exclude=["number"]).columns.to_list()
)


for col in colonnes_categorielles :
    X_train_fe[col] = X_train_fe[col].astype(str).fillna("Manquant")
    X_test_fe[col] = X_test_fe[col].astype(str).fillna("Manquant")

categories_completes = pd.concat([X_train_fe[colonnes_categorielles],
                                  X_test_fe[colonnes_categorielles]],
                                  axis=0)

for col in colonnes_categorielles:
    categories_completes[col] = categories_completes[col].astype("category")

X_train_fe[colonnes_categorielles] = categories_completes.iloc[:len(X_train_fe)][colonnes_categorielles]
X_test_fe[colonnes_categorielles] = categories_completes.iloc[:len(X_test_fe)][colonnes_categorielles]

for col in colonnes_categorielles :
    X_train_fe[col] = X_train_fe[col].cat.codes.astype("int4")
    X_test_fe[col] = X_test_fe[col].cat.codes.astype("int4")

In [ ]:
mlflow.set_experiment("tweedie_trio_optuna")

def objective(essai):

    with mlflow.start_run() :
        parametres = {
            "tweedie" : "objective",
            "tweedie_variance_power" : 1.5,
            "metrics" : "rmse",
            "learning_rate" : essai.suggest_int("learning_rate",0.01,0.15),
            "max_depth" : essai.suggest_int("max_depth",3,10),
            "num_leaves" : essai.suggest_int("num_leaves",16,128),
            "bagging_fraction" : essai.suggest_float("bagging_fraction",0.6,1.0),
            "feature_fraction" : essai.suggest_float("feature_fraction",0.6,1.0),
            "bagging_freq" : 1,
            "seed" : 42,
            "verbose" : -1
        }

        mlflow.log_params(parametres)
        X_train,X_valid,y_train,y_valid,poids_train,poids_valid = train_test_split(
            y_train_fe[variables],
            y_train_fe["PRIME_PURE"],
            y_train_fe["ANNEE_ASSURANCE"],
            test_size=0.2,
            stratify=y_train_fe["A_SINISTRE"]
        )

        donnees_train = lgb.Dataset(X_train,label=y_train,weight=poids_train)
        donnees_valid = lgb.Dataset(X_valid,label=y_valid,weight=poids_valid)

        modele = lgb.train(
            parametres,
            donnees_train,
            valid_sets=[donnees_valid],
            callbacks=[LightGBMPruningCallback(essai,"rmse"),
                       lgb.early_stopping(stopping_rounds=100,verbose=False)
                       ],
            num_boost_round=2000
        )

        predictions = modele.predict(donnees_train)

        rmse = root_mean_squared_error((y_valid - predictions),sample_weight=poids_valid)

        mlflow.lightgbm.log_model(modele,"modele")
        mlflow.metrics(rmse,"rmse")
        return rmse

essais = 100
etudes = optuna.create_study(direction="minimize")
etudes.optimize(objective(essais),n_trials=essais)

meilleurs_hyperparams = etudes.best_params
meilleurs_hyperparams.update({
    "objective": "tweedie",
    "tweedie_variance_power": 1.5,
    "metric": "rmse",
    "seed": 42,
    "verbose": -1
})

In [ ]:
NB_FOLDS = 5
k_folds = StratifiedKFold(n_splits=NB_FOLDS,shuffle=True,random_state=42)

oof_lgb = np.zeros(len(X_train_fe))
oof_catb = np.zeros(len(X_train_fe))
oof_xgb = np.zeros(len(X_train_fe))

test_lgb =  np.zeros(len(X_test_fe))
test_catb =  np.zeros(len(X_test_fe))
test_xgb =  np.zeros(len(X_test_fe))

num_FOLD = 1

In [ ]:
for idx_train, idx_valid in k_folds.split(X_train_fe,y_train_fe["A_SINISTRE"]) :
    print(f"\n Entraînement de {num_FOLD} / {NB_FOLDS} FOLD :")

    # Séparation des données
    X_train_fold = X_train_fe[variables].iloc[idx_train]
    X_valid_fold = X_test_fe[variables].iloc[idx_valid]
    y_train_fold = y_train_fe["PRIME_PURE"].iloc[idx_train]
    y_valid_fold = y_train_fe["PRIME_PURE"].iloc[idx_valid]
    poids_train_fold = y_train_fe["ANNEE_ASSURANCE"].iloc[idx_train]
    poids_valid_fold = y_train_fe["ANNEE_ASSURANCE"].iloc[idx_valid]

    #---------LightGBM---------#
    donnees_train_lgb = lgb.Dataset(X_train_fold,label=y_train_fold,weight=poids_train_fold)
    donnees_valid_lgb = lgb.Dataset(X_valid_fold,label=y_valid_fold,weight=poids_valid_fold)

    modele_lgb = lgb.train(meilleurs_hyperparams,
                           donnees_train_lgb,
                           num_boost_round=800,
                           valid_sets=[donnees_valid_lgb],
                           callbacks=[lgb.early_stopping(50,verbose=False)]
                           )
    
    predictions_lgb = modele_lgb.predict(X_valid_fold)
    oof_lgb[idx_valid] = predictions_lgb
    test_lgb += modele_lgb.predict(X_test_fe[variables]) / NB_FOLDS

    #---------CatBoost---------#
    donnees_train_catb = Pool(X_train_fold,label=y_train_fold,weight=poids_train_fold)
    donnees_valid_catb = Pool(X_valid_fold,label=y_valid_fold,weight=poids_valid_fold)

    modele_catb = CatBoostRegressor(iterations=800,
                                    learning_rate=0.08,
                                    depth=8,
                                    verbose=0,
                                    random_seed=42,
                                    colsample_bylevel=0.3,
                                    eval_metric="RMSE",
                                    loss_function="Tweedie:variance_power=1.5"
                                    )
    
    modele_catb.fit(donnees_train_catb,
                    eval_set=donnees_valid_catb,
                    early_stopping_rounds=50)
    oof_catb[idx_valid] = np.clip(modele_catb.predict(X_valid_fold),0,None)
    test_catb += np.clip(modele_catb.predict(X_test_fe[variables]),0,None) / NB_FOLDS

    #---------XGBoost---------#
    donnees_train_xgb = xgb.DMatrix(data=X_train_fold,label=y_train_fold,weight=poids_train_fold)
    donnees_valid_xgb = xgb.DMatrix(data=X_valid_fold,label=y_valid_fold,weight=poids_valid_fold)
    donnees_test_xgb = xgb.DMatrix(X_test_fe[variables])

    parametres_xgb = {
        'objective': 'reg:tweedie',
        'tweedie_variance_power': 1.5,
        'eval_metric': 'rmse',
        'learning_rate': 0.05,
        'max_depth': 5,
        'tree_method': 'hist',
        'seed': 42
    }
    modele_xgb = xgb.train(params=parametres_xgb,
                           dtrain=donnees_train_xgb,
                           num_boost_round=800,
                           evals=[(donnees_valid_xgb,"val")],
                           early_stopping_rounds=50,
                           verbose_eval = False
                           )
    oof_xgb[idx_valid] = np.clip(modele_xgb.predict(X_valid_fold), 0, None)
    test_xgb += np.clip(modele_xgb.predict(donnees_test_xgb), 0, None) / NB_FOLDS

    gc.collect()
    num_FOLD += 1



connection au drive spécificité Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/DescriptifNotebook

Mounted at /content/drive
/content/drive/MyDrive/DescriptifNotebook


## CHARGEMENT DES DONNEES DU HACKATHON

Cette fonction ci-dessous permet de charger de gros fichiers CSV par blocs (`chunks`) pour économiser la mémoire vive (RAM). Elle sélectionne aléatoirement une fraction des données (par défaut 20%) pour faciliter l'exploration et l'entraînement rapide sur un sous-ensemble du dataset.

In [ ]:
def charger_csv(chemin, fraction=0.20, taille_bloc=100_000, graine=42):
    # Générateur aléatoire
    generateur = np.random.default_rng(graine)
    morceaux_echantillonnes = []
    # Lecture par blocs du train pour baisser la RAM
    for bloc in pd.read_csv(chemin, chunksize=taille_bloc):
        masque = generateur.random(len(bloc)) < fraction
        morceaux_echantillonnes.append(bloc[masque])
    return pd.concat(morceaux_echantillonnes, ignore_index=True)

FRACTION = 0.20
X_entrainement_brut = charger_csv("train_input.csv", fraction=FRACTION)
y_entrainement_brut = charger_csv("train_output.csv", fraction=FRACTION)
X_test_complet = pd.read_csv("test_input.csv")

## FEATURE ENGINEERING

On transforme les données brutes en variables plus explicatives :
- Elle calcule le nombre de dérogations et de caractéristiques présentes, agrège les franchises, et traite les données météorologiques.
- Elle applique également une transformation logarithmique sur l'ancienneté pour normaliser sa distribution et réduit la taille mémoire des types de données pour optimiser l'usage de la RAM. (limitée à 12GO sur Google Colab)

In [ ]:
def ingenierie_de_base(df):
    df_fe = df.copy()
    # comptage du nombre de valeurs non nulles
    colonnes_derog = [col for col in df_fe.columns if 'DEROG' in col]
    df_fe['NB_DEROG'] = df_fe[colonnes_derog].notna().sum(axis=1).astype('int8')
    colonnes_caract = [col for col in df_fe.columns if 'CARACT' in col]
    df_fe['NB_CARACT'] = df_fe[colonnes_caract].notna().sum(axis=1).astype('int8')

    # Somme de FRCH1 + FRCH2
    if 'FRCH1' in df_fe.columns and 'FRCH2' in df_fe.columns:
        df_fe['FRCH_TOTAL'] = (
            pd.to_numeric(df_fe['FRCH1'], errors='coerce').fillna(0) +
            pd.to_numeric(df_fe['FRCH2'], errors='coerce').fillna(0)
        ).astype('float32')

    # Colonnes météo : celles contenant "NBJ" ou "FFM"
    colonnes_meteo = [col for col in df_fe.columns if 'NBJ' in col or 'FFM' in col]

    if len(colonnes_meteo) > 0:
        # Conversion en float32
        for col in colonnes_meteo:
            df_fe[col] = pd.to_numeric(df_fe[col], errors='coerce').astype('float32')
        # Moyenne météo
        df_fe['METEO_MOYENNE'] = df_fe[colonnes_meteo].mean(axis=1).astype('float32')
        # Maximum météo
        df_fe['METEO_MAX'] = df_fe[colonnes_meteo].max(axis=1).astype('float32')
    # Transformation logarithmique de l'ancienneté
    if 'ANCIENNETE' in df_fe.columns:
        df_fe['ANCIENNETE_LOG'] = np.log1p(
            pd.to_numeric(df_fe['ANCIENNETE'], errors='coerce').fillna(0)
        ).astype('float32')
    return df_fe

### Optimisation de la Mémoire Vive

Dans l'optique d'optimiser le plus possible l'entraînement, on allège les types.

In [ ]:
def reduire_tailles(df):
    # Conversion des colonnes float64/float32 en float plus léger
    for col in df.select_dtypes(include=['float64', 'float32']).columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    # Conversion des colonnes int64/int32 en int plus léger
    for col in df.select_dtypes(include=['int64', 'int32']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    return df

L'utilisation de `gc.collect()` et la suppression des DataFrames volumineux (`del`) sont essentielles sur Google Colab pour éviter de dépasser la limite des 12 Go de RAM lors de la manipulation de datasets d'assurance massifs.

In [ ]:
X_entrainement_fe = reduire_tailles(ingenierie_de_base(X_entrainement_brut))
X_test_fe_complet = reduire_tailles(ingenierie_de_base(X_test_complet))

# On libère la mémoire
del X_entrainement_brut
gc.collect()

107

## CIBLES

Cette section est destinée à préparer les variables cibles :
- elle redresse les montants de charge à zéro minimum et calcule la 'PRIME_PURE' en normalisant la charge par la durée d'assurance.
- On crée également une variable binaire 'A_SINISTRE' pour indiquer si un dossier a généré un coût, ce qui servira pour la stratification lors de l'entraînement.

### Logique Métier : Prime Pure
La `PRIME_PURE` représente le coût sinistre théorique pour une année complète d'assurance. En divisant la charge par la durée (`ANNEE_ASSURANCE`), nous normalisons la cible pour que le modèle apprenne un risque annuel indépendant de la durée du contrat.

In [ ]:
y_entrainement = y_entrainement_brut.copy()
y_entrainement['CHARGE'] = y_entrainement['CHARGE'].clip(lower=0) # La colonne CHARGE ne peut pas être < 0 donc minimum à 0
y_entrainement['PRIME_PURE'] = (y_entrainement['CHARGE'] / y_entrainement['ANNEE_ASSURANCE']).astype('float32') # PRIME_PURE = CHARGE / ANNEE_ASSURANCE
y_entrainement['A_SINISTRE'] = (y_entrainement['CHARGE'] > 0).astype(int) # 1 si sinistre, 0 sinon

## VARIABLES EXPLICATIVES + ENCODAGE DES CATEGORIES

- identifier les variables catégorielles,
- traiter les valeurs manquantes en les remplaçant par 'Manquant',
- convertir ces variables en codes numériques.

In [ ]:
colonnes_a_supprimer = ['ID', 'ANNEE_ASSURANCE']

# variables explicatives
variables = [col for col in X_entrainement_fe.columns if col not in colonnes_a_supprimer]
# colonnes non numériques
colonnes_categorielles = (
    X_entrainement_fe[variables]
    .select_dtypes(exclude=['number'])
    .columns
    .tolist()
)
for col in colonnes_categorielles: # Conversion des catégories en chaînes + remplacement des valeurs manquantes
    X_entrainement_fe[col] = X_entrainement_fe[col].astype(str).fillna("Manquant")
    X_test_fe_complet[col] = X_test_fe_complet[col].astype(str).fillna("Manquant")

# Concaténation train + test pour encoder
categories_completes = pd.concat(
    [X_entrainement_fe[colonnes_categorielles], X_test_fe_complet[colonnes_categorielles]],
    axis=0
)

for col in colonnes_categorielles: # Conversion en category
    categories_completes[col] = categories_completes[col].astype("category")

# Réaffectation des catégories encodées dans train et test
X_entrainement_fe[colonnes_categorielles] = categories_completes.iloc[:len(X_entrainement_fe)][colonnes_categorielles]
X_test_fe_complet[colonnes_categorielles] = categories_completes.iloc[len(X_entrainement_fe):][colonnes_categorielles]

for col in colonnes_categorielles: # conversion des catégories en codes entiers
    X_entrainement_fe[col] = X_entrainement_fe[col].cat.codes.astype("int32")
    X_test_fe_complet[col] = X_test_fe_complet[col].cat.codes.astype("int32")


In [ ]:
mlflow.set_experiment("tweedie_lgbm_optuna")

parametres_base = {
    "objective":"tweedie",
    "tweedie_variance_power":1.5,
    "metric":"rmse",
    "bagging_freq":1,
    "seed":42,
    "verbose": -1
}

def objective(essai):
    
    with mlflow.start_run():
        parametres = parametres_base.update({
            "learning_rate": essai.suggest_float("learning_rate",0.01,0.15),
            "max_depth":essai.suggest_int("max_depth",3,10),
            "num_leaves":essai.suggest_int("num_leaves",16,128),
            "bagging_fraction":essai.suggest_float("bagging_fraction",0.6,1.0),
            "feature_fraction":essai.suggest_float("feature_fraction",0.6,1.0)
            })
        mlflow.log_params(parametres)
        X_train,X_valid,y_train,y_valid,poids_train,poids_valid = train_test_split(y_entrainement[variables],
                                                                                   y_entrainement["PRIME_PURE"],
                                                                                   y_entrainement["ANNEE_ASSURANCE"],
                                                                                   test_size=0.2,
                                                                                   random_state=42,
                                                                                   stratify=y_entrainement["A_SINISTRE"]
                                                                                   )
        donnees_train = lgb.Dataset(X_train,label=y_train,weight=poids_train)
        donnees_valid = lgb.Dataset(X_valid,label=y_valid,weight=poids_valid)

        modele = lgb.train(
            parametres,
            donnees_train,
            sets_valid=[donnees_valid],
            callbacks=[LightGBMPruningCallback(essai,"rmse"),
                       lgb.early_stopping(stopping_rounds=100,verbose=False)],
            num_boost_round=2000
        )

        predictions = modele.predict(donnees_valid)

        rmse = np.sqrt((y_valid - predictions)**2, weight = poids_valid)
        mlflow.log_metric("rmse",rmse)
        mlflow.lightgbm.log_model(parametres)
        return rmse

etude = optuna.create_study(direction="minimize")
etude.optimize(objective,trials=100)

meilleurs_hyperparam = etude.best_params

KeyboardInterrupt: 

## OPTIMISATION DES HYPERPARAMETRES (Optuna + MLflow)

Cette section utilise **Optuna** pour automatiser la recherche des meilleurs hyperparamètres du modèle LightGBM :
- On définit une fonction `objectif` qui entraîne un modèle avec des paramètres suggérés et retourne le RMSE.
- L'objectif est de minimiser l'erreur sur un jeu de validation stratifié.
- **MLflow** est utilisé en parallèle pour enregistrer chaque essai, permettant de visualiser les courbes d'apprentissage et de versionner les modèles.
- Le paramètre `tweedie` est choisi car il est particulièrement adapté aux données d'assurance présentant de nombreux zéros (pas de sinistre) et quelques valeurs élevées.

In [ ]:
mlflow.set_experiment("tweedie_lgbm_optuna")

def objectif(essai):
    # On run MLflow pour suivre les paramètres et métriques
    with mlflow.start_run():
        # Params LightGBM optimisés par Optuna
        parametres = {
            "objective": "tweedie",
            "tweedie_variance_power": 1.5,
            "metric": "rmse",
            "learning_rate": essai.suggest_float("learning_rate", 0.01, 0.15),
            "num_leaves": essai.suggest_int("num_leaves", 16, 128),
            "max_depth": essai.suggest_int("max_depth", 3, 10),
            "feature_fraction": essai.suggest_float("feature_fraction", 0.6, 1.0),
            "bagging_fraction": essai.suggest_float("bagging_fraction", 0.6, 1.0),
            "bagging_freq": 1,
            "seed": 42,
            "verbose": -1
        }
        # Enregistrement des paramètres dans MLflow
        mlflow.log_params(parametres)
        # Séparation entraînement / validation
        X_train, X_valid, y_train_local, y_valid_local, poids_train, poids_valid = train_test_split(
            X_entrainement_fe[variables],
            y_entrainement["PRIME_PURE"],
            y_entrainement["ANNEE_ASSURANCE"],
            test_size=0.2,
            random_state=42,
            stratify=y_entrainement["A_SINISTRE"]
        )
        # Datas LightGBM
        donnees_train = lgb.Dataset(X_train, label=y_train_local, weight=poids_train)
        donnees_valid = lgb.Dataset(X_valid, label=y_valid_local, weight=poids_valid)

        # Entraînement LightGBM avec Optuna
        modele = lgb.train(
            parametres,
            donnees_train,
            valid_sets=[donnees_valid],
            callbacks=[
                LightGBMPruningCallback(essai, "rmse"),
                lgb.early_stopping(stopping_rounds=100, verbose=False)
            ],
            num_boost_round=2000
        )
        # Prédictions validation
        predictions = modele.predict(X_valid)
        # RMSE
        rmse = np.sqrt(mean_squared_error(
            y_valid_local * poids_valid,
            predictions * poids_valid
        ))
        # Enregistrement métrique + modèle dans MLflow
        mlflow.log_metric("rmse", rmse)
        mlflow.lightgbm.log_model(modele, "modele")
        return rmse


etude = optuna.create_study(direction="minimize")
etude.optimize(objectif, n_trials=100)

meilleurs_parametres = etude.best_params
meilleurs_parametres.update({
    "objective": "tweedie",
    "tweedie_variance_power": 1.5,
    "metric": "rmse",
    "seed": 42,
    "verbose": -1
})

[I 2026-04-03 18:38:32,222] A new study created in memory with name: no-name-9e7f2e2e-4b5a-43cc-8477-77f146bcf6d2
2026/04/03 18:39:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
[I 2026-04-03 18:39:14,847] Trial 0 finished with value: 6649.662646619786 and parameters: {'learning_rate': 0.09219052096391518, 'num_leaves': 38, 'max_depth': 8, 'feature_fraction': 0.7396562372150017, 'bagging_fraction': 0.6847048931346995}. Best is trial 0 with value: 6649.662646619786.
2026/04/03 18:39:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
[I 2026-04-03 18:39:32,048] Trial 1 finished with value: 6665.237572638483 and parameters: {'learning_rate': 0.014886627196763053, 'num_leaves': 90, 'max_depth': 4, 'feature_fraction': 0.6845757246905694, 'bagging_fraction': 0.8112799954628639}. Best is trial 0 with value: 6649.662646619786.
2026/04/03 18:39:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please 

## ENTRAINEMENT AVEC VALIDATION CROISEE + ENSEMBLE

Cette section met en œuvre une validation croisée stratifiée en 5 plis (`StratifiedKFold`) pour assurer la robustesse des modèles.

Pour chaque pli, nous entraînons un ensemble composé de trois algorithmes de Boosting majeurs :
- **LightGBM** : Utilisé sa rapidité et son efficacité sur de grands jeux de données grâce au 'Leaf-wise growth'.
- **CatBoost** : Particulièrement performant pour gérer nativement les variables catégorielles (très présentes dans ce dataset via les codes DEROG/CARACT) et limiter le sur-apprentissage.
- **XGBoost** : Reconnu pour sa précision et sa gestion efficace des valeurs manquantes.

**Justification du choix par rapport aux données :**
Les données d'assurance sont typiquement tabulaires, hétérogènes et comportent souvent des relations non-linéaires complexes que les modèles linéaires classiques peinent à capturer. Le Gradient Boosting est surement la meilleure approche pour ce type de structure. L'utilisation d'un **Ensemble** permet de réduire la variance : en moyennant les prédictions, on atténue les erreurs spécifiques à chaque algorithme, ce qui améliore la généralisation sur les données de test non vues.

### Stratégie de Validation Stratifiée
Étant donné que la majorité des lignes ne comportent pas de sinistre (zéro-inflation), nous utilisons une `StratifiedKFold` basée sur la variable binaire `A_SINISTRE` pour garantir que chaque pli (fold) contient une proportion identique de dossiers avec et sans sinistres.

In [ ]:
NB_FOLDS = 5
kfold = StratifiedKFold(n_splits=NB_FOLDS, shuffle=True, random_state=42)

# Prédictions pour chaque modèle
oof_lgb = np.zeros(len(X_entrainement_fe))
oof_cat = np.zeros(len(X_entrainement_fe))
oof_xgb = np.zeros(len(X_entrainement_fe))

# Prédictions sur le test pour chaque modèle
test_lgb = np.zeros(len(X_test_fe_complet))
test_cat = np.zeros(len(X_test_fe_complet))
test_xgb = np.zeros(len(X_test_fe_complet))

numero_fold = 1

### Comparaison des Modèles de l'Ensemble
- **LightGBM** : Rapide, optimisé pour les grands volumes.
- **CatBoost** : Excellente gestion native des variables catégorielles (codes DEROG/CARACT).
- **XGBoost** : Apporte une régularisation complémentaire pour stabiliser la prédiction finale.

In [ ]:
for idx_train, idx_valid in kfold.split(X_entrainement_fe, y_entrainement['A_SINISTRE']):
    print(f"\n--- FOLD {numero_fold}/{NB_FOLDS} ---")

    # Séparation des données
    X_train_fold = X_entrainement_fe[variables].iloc[idx_train]
    X_valid_fold = X_entrainement_fe[variables].iloc[idx_valid]
    y_train_fold = y_entrainement['PRIME_PURE'].iloc[idx_train]
    y_valid_fold = y_entrainement['PRIME_PURE'].iloc[idx_valid]
    poids_train_fold = y_entrainement['ANNEE_ASSURANCE'].iloc[idx_train]
    poids_valid_fold = y_entrainement['ANNEE_ASSURANCE'].iloc[idx_valid]

    # =========================
    # LIGHTGBM
    # =========================
    print("Entraînement LightGBM...")
    donnees_train_lgb = lgb.Dataset(X_train_fold, label=y_train_fold, weight=poids_train_fold)
    donnees_valid_lgb = lgb.Dataset(X_valid_fold, label=y_valid_fold, weight=poids_valid_fold)
    modele_lgb = lgb.train(
        meilleurs_parametres,
        donnees_train_lgb,
        num_boost_round=800,
        valid_sets=[donnees_valid_lgb],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    oof_lgb[idx_valid] = modele_lgb.predict(X_valid_fold)
    test_lgb += modele_lgb.predict(X_test_fe_complet[variables]) / NB_FOLDS

    # =========================
    # CATBOOST
    # =========================
    print("Entraînement CatBoost...")
    pool_train_cat = Pool(X_train_fold, label=y_train_fold, weight=poids_train_fold)
    pool_valid_cat = Pool(X_valid_fold, label=y_valid_fold, weight=poids_valid_fold)
    modele_cat = CatBoostRegressor(
        loss_function='Tweedie:variance_power=1.5',
        eval_metric='RMSE',
        learning_rate=0.08,
        depth=5,
        colsample_bylevel=0.3,
        iterations=800,
        random_seed=42,
        verbose=0
    )
    modele_cat.fit(pool_train_cat, eval_set=pool_valid_cat, early_stopping_rounds=50)
    oof_cat[idx_valid] = np.clip(modele_cat.predict(X_valid_fold), 0, None)
    test_cat += np.clip(modele_cat.predict(X_test_fe_complet[variables]), 0, None) / NB_FOLDS

    # =========================
    # XGBOOST
    # =========================
    print("Entraînement XGBoost...")
    dtrain_xgb = xgb.DMatrix(X_train_fold, label=y_train_fold, weight=poids_train_fold)
    dvalid_xgb = xgb.DMatrix(X_valid_fold, label=y_valid_fold, weight=poids_valid_fold)
    dtest_xgb  = xgb.DMatrix(X_test_fe_complet[variables])
    parametres_xgb = {
        'objective': 'reg:tweedie',
        'tweedie_variance_power': 1.5,
        'eval_metric': 'rmse',
        'learning_rate': 0.05,
        'max_depth': 5,
        'tree_method': 'hist',
        'seed': 42
    }
    modele_xgb = xgb.train(
        parametres_xgb,
        dtrain_xgb,
        num_boost_round=800,
        evals=[(dvalid_xgb, 'val')],
        early_stopping_rounds=50,
        verbose_eval=False
    )
    oof_xgb[idx_valid] = np.clip(modele_xgb.predict(dvalid_xgb), 0, None)
    test_xgb += np.clip(modele_xgb.predict(dtest_xgb), 0, None) / NB_FOLDS

    gc.collect()
    numero_fold += 1



--- FOLD 1/5 ---
Entraînement LightGBM...
Entraînement CatBoost...
Entraînement XGBoost...

--- FOLD 2/5 ---
Entraînement LightGBM...
Entraînement CatBoost...
Entraînement XGBoost...

--- FOLD 3/5 ---
Entraînement LightGBM...
Entraînement CatBoost...
Entraînement XGBoost...

--- FOLD 4/5 ---
Entraînement LightGBM...
Entraînement CatBoost...
Entraînement XGBoost...

--- FOLD 5/5 ---
Entraînement LightGBM...
Entraînement CatBoost...
Entraînement XGBoost...


## CALCUL DU RMSE LOCAL + CREATION DU FICHIER DE SOUMISSION

- On calcule la moyenne des prédictions de l'ensemble (LGBM, CatBoost, XGBoost) et évalue la performance via le RMSE local.
- Les résultats sont ensuite convertis en montants de charge et formatés selon les exigences du hackathon.
- Le fichier CSV généré est prêt pour la soumission officielle sur la plateforme Challenge Data.

In [ ]:
# Moyenne des prédictions OOF des trois modèles
oof_prime_pure = (oof_lgb + oof_cat + oof_xgb) / 3

# Conversion en CHARGE
oof_charge = oof_prime_pure * y_entrainement['ANNEE_ASSURANCE']

# RMSE local sur la charge
rmse_local = np.sqrt(mean_squared_error(y_entrainement['CHARGE'], oof_charge))
print("RMSE LOCAL GLOBAL :", rmse_local)

# Moyenne des prédictions test des trois modèles
test_prime_pure = (test_lgb + test_cat + test_xgb) / 3

# Conversion en CHARGE pour le test
test_charge = test_prime_pure * X_test_fe_complet['ANNEE_ASSURANCE']

# y_test final
soumission = pd.DataFrame({
    'ID': X_test_fe_complet['ID'],
    'ANNEE_ASSURANCE': X_test_fe_complet['ANNEE_ASSURANCE'],
    'FREQ': 0,   # Colonnes imposées par le format de soumission
    'CM': 0,
    'CHARGE': test_charge
})

# Sauvegarde
soumission.to_csv("soumission_trio_optuna_mlflow_3.csv", index=False)


RMSE LOCAL GLOBAL : 6964.173320329926
Fichier de soumission enregistré : soumission_trio_optuna_mlflow_3.csv
